# Encounter Map

Search for the current user's most recent encounters and plot their
locations on an interactive map.

**Extra dependency** — install before running this notebook:

```bash
uv pip install ipyleaflet
```

Set the usual environment variables (`WILDBOOK_URL`, `WILDBOOK_USERNAME`,
`WILDBOOK_PASSWORD`) before starting the kernel, or pass credentials
explicitly to `client.login()`.

In [ ]:
from pywildbook import WildbookClient
from ipyleaflet import Map, Marker, Popup, MarkerCluster

In [ ]:
client = WildbookClient()
user = client.login()
print(f"Logged in as {user['username']}")

In [ ]:
results = client.search_encounters(
    client.filter_current_user(),
    size=50,
    sort='date',
    sort_order='desc'
)

encounters = results.get('hits', [])
print(f"Found {len(encounters)} encounters")

In [ ]:
def _popup(enc):
    """Return a Popup populated with encounter details."""
    genus = enc.get('genus', '')
    species = enc.get('specificEpithet', '')
    name = f"{genus} {species}".strip() or 'Unknown species'
    return Popup(
        f"<b>{name}</b><br>"
        f"Year: {enc.get('year', 'N/A')}<br>"
        f"Location: {enc.get('verbatimLocality', 'N/A')}<br>"
        f"<small>ID: {enc.get('id', '')}</small>",
        max_width=250
    )


# Keep only encounters that carry a geo point
mapped = [
    e for e in encounters
    if isinstance(e.get('locationGeoPoint'), dict)
    and 'lat' in e['locationGeoPoint']
    and 'lon' in e['locationGeoPoint']
]
print(f"{len(mapped)} of {len(encounters)} encounters have a location")

# Center on the mean of plotted points, or fall back to world view
if mapped:
    lats = [e['locationGeoPoint']['lat'] for e in mapped]
    lons = [e['locationGeoPoint']['lon'] for e in mapped]
    center = [sum(lats) / len(lats), sum(lons) / len(lons)]
else:
    center = [0, 0]

m = Map(center=center, zoom=4)

if mapped:
    markers = [
        Marker(
            location=[e['locationGeoPoint']['lat'], e['locationGeoPoint']['lon']],
            popup=_popup(e)
        )
        for e in mapped
    ]
    m.add_layer(MarkerCluster(markers=markers))

m

In [ ]:
client.logout()